# Whose pin does Stockfish 18 carry?

Six labels over the same 151,614 positions, probed at one layer — `acc||tacc`, the full
pre-nonlinearity state and the layer that carried the Lichess tag best in `probe_mlp.ipynb`.

Three labels are **cook.py's own** — the program that produced the Lichess `pin` theme,
run per position by `../cook_pin.py` (identical to the vendored source on all 6,100,960
puzzles in the export). Three are **this repo's**, from `pin_relevance.py`.

| label | what fires it | positive |
|---|---|---|
| `cook.pin` | the tag itself: `attack or escape`, over the boards after each solver move | 28.71% |
| `cook.attack` | a pinned enemy piece may not take a pov piece worth more, or hanging | 26.06% |
| `cook.escape` | an attacker on the pin ray hits a pinned piece that may not step aside | 3.95% |
| `ours.D` | the solver's move lands on a square the pinned piece may not contest | 25.60% |
| `ours.S` | ...and nothing else guards that square | 18.79% |
| `ours.A` | the solver's move attacks the pinned piece | 37.54% |

cook's three are properties of a **position** and never mention the solver's move. Ours are
properties of a **move**. Same probe, same folds, same activations: a difference between
bars is a difference in the label.

## Provenance

| File | Role |
|---|---|
| `../tools/lichess-puzzler/fetch.sh` | vendors `ornicar/lichess-puzzler` at the pinned commit |
| `../cook_pin.py` | cook.py's two pin predicates per position; `selftest` proves the identity |
| `auto_labels.py` | the six labels, aligned to `meta.tsv` -> `data/labels.npz` |
| `probe_lib.py` | the probe, the folds, the row cache |
| `sweep_labels.py` | one worker: build the layer once, train every label on it |

```bash
../tools/lichess-puzzler/fetch.sh
zstdcat lichess_db_puzzle.csv.zst | python ../cook_pin.py selftest   # 0 mismatches
zstdcat lichess_db_puzzle.csv.zst | python auto_labels.py
OMP_NUM_THREADS=8 python sweep_labels.py A 'acc||tacc'
```

Folds are a plain `KFold`, not stratified: stratified folds depend on `y`, so six labels
would get six partitions and the bars would not be measured on the same split. That is the
one departure from `probe_mlp.ipynb`, whose `acc||tacc` number (0.839 for the exported tag)
is therefore close to but not identical to `cook.pin` here.

## 1 · Configuration

In [ ]:
from pathlib import Path
import os, sys, json, time, gc

_pylibs = Path.cwd().parent / ".pylibs"
if _pylibs.is_dir() and str(_pylibs) not in sys.path:
    sys.path.insert(0, str(_pylibs))
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import probe_lib as P
from sweep_labels import layer_matrix, acts

LAYER    = "acc||tacc"            # artifacts are named after it, as in distance_mlp.ipynb
SLUG     = LAYER.replace("||", "_")
HERE     = Path.cwd()
CACHE    = P.DATA / "labels_nb.json"
OUT_JSON = HERE / "results_labels.json"
OUT_CSV  = HERE / "results_labels.csv"
OUT_PNG  = HERE / "auto_labels.png"

os.environ.setdefault("OMP_NUM_THREADS", str(os.cpu_count()))
print(f"layer {LAYER}   probe MLP{P.MLP['hidden_layer_sizes']}   {P.FOLDS}-fold KFold, seed {P.SEED}")

## 2 · The six labels

In [ ]:
y, NAMES, FEAT, FEAT_NAMES = P.labels()
N = len(y)
SPLITS = P.splits(N)
assert N == len(acts), f"{len(acts)} dump records vs {N} labels"

print(f"N = {N:,}\n")
print(f"{'label':14}{'positives':>11}{'rate':>8}")
for j, nm in enumerate(NAMES):
    print(f"{nm:14}{y[:, j].sum():>11,}{y[:, j].mean()*100:>7.2f}%")

print(f"\nagreement between labels (%)\n{'':14}" + "".join(f"{n.split('.')[1]:>9}" for n in NAMES))
for i, a in enumerate(NAMES):
    print(f"{a:14}" + "".join(f"{(y[:, i] == y[:, k]).mean()*100:9.2f}" for k in range(len(NAMES))))

## 3 · The probe

Rows measured by `sweep_labels.py` are read from `data/labels_*.json`; anything missing is
trained here. The layer matrix is built once and reused across labels.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

todo = [nm for nm in NAMES if f"{nm} {LAYER}" not in P.cached()]
X = layer_matrix(LAYER) if todo else None
rows = {nm: P.measure(f"{nm} {LAYER}", lambda: X, y[:, j], SPLITS, CACHE,
                      layer=LAYER, label=nm)
        for j, nm in enumerate(NAMES)}
del X; gc.collect()

## 4 · Two floors under every bar

**Metadata.** Rating, number of solver moves, in-check, material. No board, no network.
It is the control for the puzzle-length confound: "fires on at least one solver move" is
monotone in the number of solver moves, and that alone scores 0.607 for `ours.A` against
0.44-0.46 for the others, so part of any `ours.A` bar is puzzle length rather than pin.

**Shuffled label.** The same probe on a permutation. A misaligned join and a property the
network does not carry both look like 0.5, so the floor is measured, not assumed. One row
is enough: a permutation destroys the label the same way whatever the label was.

In [ ]:
floors = {nm: P.measure(f"{nm} @meta", lambda: FEAT, y[:, j], SPLITS, CACHE,
                        layer="@meta", label=nm)
          for j, nm in enumerate(NAMES)}

rng = np.random.default_rng(P.SEED)
perm = rng.permutation(N)
Xs = layer_matrix(LAYER) if f"cook.pin @shuffled" not in P.cached() else None
shuffled = P.measure("cook.pin @shuffled", lambda: Xs, y[perm, 0], SPLITS, CACHE,
                     layer="@shuffled", label="cook.pin")
del Xs; gc.collect()

print(f"\n{'label':14}{'metadata':>10}{LAYER:>12}{'gain':>8}")
for nm in NAMES:
    g = rows[nm]["auroc"] - floors[nm]["auroc"]
    print(f"{nm:14}{floors[nm]['auroc']:>10.3f}{rows[nm]['auroc']:>12.3f}{g:>+8.3f}")
print(f"\nshuffled label at {LAYER}: {shuffled['auroc']:.3f}")

## 5 · Results

In [ ]:
import csv
meta = dict(n=int(N), layer=LAYER, folds=P.FOLDS, seed=P.SEED, labels=NAMES,
            rates={nm: float(y[:, j].mean()) for j, nm in enumerate(NAMES)},
            mlp={k: str(v) for k, v in P.MLP.items()},
            auroc={nm: rows[nm]["auroc"] for nm in NAMES},
            meta_floor={nm: floors[nm]["auroc"] for nm in NAMES},
            shuffled=shuffled["auroc"])
all_rows = list(rows.values()) + list(floors.values()) + [shuffled]
json.dump(dict(meta=meta, rows=all_rows), open(OUT_JSON, "w"), indent=1)
with open(OUT_CSV, "w", newline="") as f:
    fn = ["key", "label", "layer", "dims", "auroc", "sd", "acc", "epochs", "secs"]
    w = csv.DictWriter(f, fieldnames=fn); w.writeheader()
    w.writerows({k: r.get(k) for k in fn} for r in all_rows)
print(f"wrote {OUT_JSON.name}, {OUT_CSV.name}")

## 6 · Plot

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#dedcd6"
WARM, COOL = ["#8a3d0d", "#c4711f", "#e0a458"], ["#0d366b", "#2a78d6", "#7fb2e8"]
COLOR = dict(zip(NAMES, WARM + COOL))
WHAT = {
    "cook.pin":    "the tag: attack or escape",
    "cook.attack": "pinned piece may not take",
    "cook.escape": "pinned piece may not flee",
    "ours.D":      "move lands where it can't be contested",
    "ours.S":      "...and nothing else guards there",
    "ours.A":      "move attacks the pinned piece",
}
order = NAMES[::-1]                                  # top-to-bottom = cook first
ypos = np.arange(len(order))
v  = np.array([rows[nm]["auroc"] for nm in order])
sd = np.array([rows[nm]["sd"] for nm in order])
fl = np.array([floors[nm]["auroc"] for nm in order])
sh = shuffled["auroc"]

fig, ax = plt.subplots(figsize=(12.4, 5.6))
fig.patch.set_facecolor("white"); ax.set_facecolor("white")

ax.barh(ypos, v - 0.5, left=0.5, height=0.62,
        color=[COLOR[nm] for nm in order], zorder=3)
ax.errorbar(v, ypos, xerr=sd, fmt="none", ecolor=INK, elinewidth=0.9, capsize=3,
            alpha=.55, zorder=5)

for i, nm in enumerate(order):
    ax.plot([fl[i], fl[i]], [i - 0.31, i + 0.31], color="#ffffff", lw=3.2, zorder=6)
    ax.plot([fl[i], fl[i]], [i - 0.31, i + 0.31], color=INK, lw=1.4, zorder=7)
    ax.text(v[i] + sd[i] + 0.004, i, f"{v[i]:.3f}", va="center", ha="left",
            fontsize=10.5, color=INK, weight="bold", zorder=8)
    ax.text(0.507, i + 0.30, WHAT[nm], va="bottom", ha="left", fontsize=8.6,
            color="#ffffff", style="italic", zorder=8)

ax.axvline(sh, color="#8a8a84", lw=1.3, ls=":", zorder=4)
ax.text(sh, len(order) - 0.32, f" shuffled label {sh:.3f} ", fontsize=8.8,
        color=MUTED, va="bottom", ha="left")
ax.text(fl.max() + 0.004, -0.92, "black tick = metadata-only floor (rating, solver moves, "
        "in check, material)", fontsize=8.8, color=MUTED, va="center", ha="left")

ax.set_yticks(ypos)
ax.set_yticklabels([f"{nm}\n{y[:, NAMES.index(nm)].mean()*100:.1f}% +" for nm in order],
                   fontsize=10, color=INK)
ax.set_xlabel("ROC AUC   (5-fold CV, mean ± sd)", fontsize=10.5, color=INK)
ax.set_xlim(0.5, max(v.max() + 0.06, 0.90))
ax.set_ylim(-1.4, len(order) - 0.35)
ax.tick_params(axis="x", labelsize=9.5, colors=MUTED, length=0)
ax.tick_params(axis="y", length=0)
ax.grid(axis="x", color=GRID, lw=0.7, zorder=1); ax.set_axisbelow(True)
for s_ in ("top", "right", "bottom"): ax.spines[s_].set_visible(False)
ax.spines["left"].set_color(GRID)

ax.text(0, 1.16, "Whose pin does Stockfish 18 carry?", transform=ax.transAxes,
        fontsize=14, color=INK, va="bottom", weight="bold")
ax.text(0, 1.015,
        f"One layer — {LAYER}, the full pre-nonlinearity state — and six definitions of "
        f"'this pin matters', on the same {N:,} positions\n"
        f"warm = cook.py's own rule, the program behind the Lichess `pin` tag, a property of "
        f"positions the solver never faces · cool = this repo's move-bound rules\n"
        f"probe = MLP{P.MLP['hidden_layer_sizes']} held identical · {P.FOLDS}-fold KFold, one "
        f"partition for every label · axis starts at chance (0.500)",
        transform=ax.transAxes, fontsize=9.5, color=MUTED, va="bottom", linespacing=1.5)

fig.savefig(OUT_PNG, dpi=170, bbox_inches="tight", facecolor="white")
print(f"wrote {OUT_PNG}")
plt.close(fig)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUT_PNG)))

## 7 · Reading it